# 00 · Wine Quality — Data Acquisition & Cleaning

## Part 0 — Acquire & Clean (Wine Quality)

A change of pace from the three time-series practices: this is a **cross-sectional** dataset — 6,497
wines (1,599 red + 4,898 white from the UCI Wine Quality data), each described by **11
physicochemical measurements** and rated by tasters on an ordinal **`quality`** score (3–9). The
goal will be to predict and explain quality.

Two features of this dataset make it interesting (and a little dangerous):
- the **target is ordinal and heavily imbalanced** (most wines are average; the extremes are rare),
- there are **1,177 duplicate rows (18%)** — which, if split carelessly, leak test answers into training.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = pathlib.Path.cwd(); ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import data, eda
eda.set_style()
pd.set_option("display.width", 120, "display.max_columns", 30)
print("setup ok | numpy", np.__version__, "| pandas", pd.__version__)


setup ok | numpy 2.1.3 | pandas 2.3.3


### 1. Structure

In [2]:
df = data.clean()
print("shape:", df.shape, "| missing values:", int(df.isna().sum().sum()))
print("wine types:", df.wine_type.value_counts().to_dict())
print("\n11 features + ordinal quality target:")
print(df.dtypes.to_string())

shape: (6497, 13) | missing values: 0
wine types: {'white': 4898, 'red': 1599}

11 features + ordinal quality target:
fixed_acidity            float64
volatile_acidity         float64
citric_acid              float64
residual_sugar           float64
chlorides                float64
free_sulfur_dioxide      float64
total_sulfur_dioxide     float64
density                  float64
pH                       float64
sulphates                float64
alcohol                  float64
quality                    int64
wine_type               category


### 2. The duplicate-rows trap

18% of rows are exact duplicates — wines with identical measurements (plausibly repeat lab readings
or wines from the same batch). They're harmless for *describing* the data, but for *modelling* they
are a leakage risk: a duplicate landing in both train and test lets a model "memorise" a test answer.
So we **keep them for EDA** but expose a `dedup()` to use **before** any train/test split.

In [3]:
print("duplicate rows:", int(df.duplicated().sum()), "(%.0f%%)" % (100*df.duplicated().mean()))
print("after dedup:", data.dedup(df).shape[0], "wines")
print("\nexample of a repeated wine (first duplicated row appears %d times):"
      % int((df == df[df.duplicated(keep=False)].iloc[0]).all(axis=1).sum()))
df[df.duplicated(keep=False)].sort_values(data.NUMERIC).head(2)[data.NUMERIC[:5] + ["quality"]]

duplicate rows: 1177 (18%)
after dedup: 5320 wines

example of a repeated wine (first duplicated row appears 2 times):


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,quality
6270,4.9,0.335,0.14,1.3,0.036,5
6271,4.9,0.335,0.14,1.3,0.036,5


### 3. The target — ordinal and imbalanced

`quality` is an **ordinal** label (6 > 5 is meaningfully ordered, unlike nominal categories) and
badly **imbalanced**: the middle grades dominate, the best and worst wines are rare. Quality 9 has
just **5** wines in the whole dataset — a real challenge for any classifier.

In [4]:
print(df.quality.value_counts().sort_index().to_string())
print("\ntop class share: %.0f%% | extremes (3,4,8,9): %.1f%% | quality 9: %d wines"
      % (100*df.quality.value_counts(normalize=True).max(), 100*df.quality.isin([3,4,8,9]).mean(), int((df.quality==9).sum())))
print("\ncoarse 3-band framing (still imbalanced):")
print(data.quality_band(df.quality).value_counts().sort_index().to_string())

quality
3      30
4     216
5    2138
6    2836
7    1079
8     193
9       5

top class share: 44% | extremes (3,4,8,9): 6.8% | quality 9: 5 wines

coarse 3-band framing (still imbalanced):
quality
low (<=4)      246
mid (5-6)     4974
high (>=7)    1277


### 4. Clean & persist

In [5]:
data.build_processed()
print("wrote data/processed/wine_clean.csv — Part 0 complete (duplicates kept + flagged; dedup() ready for modelling).")

wrote data/processed/wine_clean.csv — Part 0 complete (duplicates kept + flagged; dedup() ready for modelling).
